# Phase 8 — Machine Learning Model Development

**Goal:** train and validate four regression models using the same leakage-safe expanding-window walk-forward folds.

Each model has its **own complete code block**. No shared model-training function is used.

**Run control:** set one or more `RUN_...` flags to `True`. By default they are `False` so the notebook does not accidentally start long training jobs.

**Important:** the final January–February 2017 holdout is not used in this phase.

## 1. Load the walk-forward folds

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "project_config.py").is_file():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT))

from src.project_config import *
from src.modeling_utils import *

import numpy as np
import pandas as pd

manifest_path = SPLIT_DIR / "walk_forward_manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError("Run 06_data_split.ipynb first.")

manifest = pd.read_csv(manifest_path)
required_manifest_columns = {"fold", "train_file", "validation_file", "train_rows", "validation_rows"}
missing_manifest = required_manifest_columns - set(manifest.columns)
if missing_manifest:
    raise ValueError(f"Walk-forward manifest is missing columns: {sorted(missing_manifest)}")

display(manifest[["fold", "train_rows", "validation_rows", "train_file", "validation_file"]])

,fold,train_rows,validation_rows,train_file,validation_file
0,fold_1,82553,87927,fold_1_train_2013_to_2013.csv,fold_1_validation_2014.csv
1,fold_2,170480,87088,fold_2_train_2013_to_2014.csv,fold_2_validation_2015.csv
2,fold_3,257568,86826,fold_3_train_2013_to_2015.csv,fold_3_validation_2016.csv


## 2. Define the model inputs

In [3]:
TARGET_COLUMN = TARGET
DROP_COLUMNS = ["No", "datetime", "target_datetime", "has_next_hour", TARGET_COLUMN]

example_train = pd.read_csv(
    SPLIT_DIR / str(manifest.loc[0, "train_file"])
)
FEATURE_COLUMNS = [c for c in example_train.columns if c not in DROP_COLUMNS]

if not FEATURE_COLUMNS:
    raise ValueError("No model features were found after removing target/metadata columns.")

print("Total model features:", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)

Total model features: 33
['year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station', 'day_of_week', 'is_weekend', 'season', 'hour_sin', 'hour_cos', 'PM2.5_lag_1h', 'PM2.5_lag_2h', 'PM2.5_lag_3h', 'PM2.5_lag_6h', 'PM2.5_lag_12h', 'PM2.5_lag_24h', 'PM2.5_roll_mean_3h', 'PM2.5_roll_mean_6h', 'PM2.5_roll_mean_24h', 'PM2.5_roll_std_6h', 'PM2.5_roll_std_24h']


## 3. Model 1 — Linear Regression

**Purpose:** provide a simple linear reference model.

**Preprocessing:** training-only IQR capping → median imputation → one-hot encoding → standardization.

In [4]:
# Independent Linear Regression block
RUN_LINEAR_REGRESSION = True
linear_results = None

if RUN_LINEAR_REGRESSION:
    fold_results = []

    for _, item in manifest.iterrows():
        train = pd.read_csv(SPLIT_DIR / item["train_file"])
        validation = pd.read_csv(SPLIT_DIR / item["validation_file"])

        columns, numeric, categorical = model_input_columns(
            train, TARGET_COLUMN, feature_columns=FEATURE_COLUMNS
        )
        X_train = train[columns]
        y_train = train[TARGET_COLUMN]
        X_validation = validation[columns]
        y_validation = validation[TARGET_COLUMN]

        pipeline = make_model_pipeline("Linear Regression", numeric, categorical)
        pipeline.fit(X_train, y_train)

        train_prediction = pipeline.predict(X_train)
        validation_prediction = pipeline.predict(X_validation)
        train_metrics = regression_metrics(y_train, train_prediction)
        validation_metrics = regression_metrics(y_validation, validation_prediction)

        fold_results.append({
            "model": "Linear Regression",
            "fold": item["fold"],
            "train_rows": len(train),
            "validation_rows": len(validation),
            "train_MAE": train_metrics["MAE"],
            "validation_MAE": validation_metrics["MAE"],
            "train_RMSE": train_metrics["RMSE"],
            "validation_RMSE": validation_metrics["RMSE"],
            "train_R2": train_metrics["R2"],
            "validation_R2": validation_metrics["R2"],
        })

    linear_results = pd.DataFrame(fold_results)
    display(linear_results.round(4))
else:
    print("Linear Regression skipped. Set RUN_LINEAR_REGRESSION = True to run it.")

,model,fold,train_rows,validation_rows,train_MAE,validation_MAE,train_RMSE,validation_RMSE,train_R2,validation_R2
0,Linear Regression,fold_1,82553,87927,12.7963,13.1769,22.6441,25.2254,0.9047,0.8975
1,Linear Regression,fold_2,170480,87088,12.7759,13.5414,23.3933,29.2137,0.9058,0.8799
2,Linear Regression,fold_3,257568,86826,13.7164,13.1168,25.2042,24.6555,0.8983,0.8944


## 4. Model 2 — Decision Tree Regressor

**Purpose:** learn non-linear rules and feature interactions.

**Overfitting controls:** bounded depth and minimum leaf size.

In [5]:
# Independent Decision Tree block
RUN_DECISION_TREE = True
decision_tree_results = None

if RUN_DECISION_TREE:
    fold_results = []

    for _, item in manifest.iterrows():
        train = pd.read_csv(SPLIT_DIR / item["train_file"])
        validation = pd.read_csv(SPLIT_DIR / item["validation_file"])

        columns, numeric, categorical = model_input_columns(
            train, TARGET_COLUMN, feature_columns=FEATURE_COLUMNS
        )
        X_train = train[columns]
        y_train = train[TARGET_COLUMN]
        X_validation = validation[columns]
        y_validation = validation[TARGET_COLUMN]

        pipeline = make_model_pipeline("Decision Tree", numeric, categorical)
        pipeline.fit(X_train, y_train)

        train_prediction = pipeline.predict(X_train)
        validation_prediction = pipeline.predict(X_validation)
        train_metrics = regression_metrics(y_train, train_prediction)
        validation_metrics = regression_metrics(y_validation, validation_prediction)

        fold_results.append({
            "model": "Decision Tree",
            "fold": item["fold"],
            "train_rows": len(train),
            "validation_rows": len(validation),
            "train_MAE": train_metrics["MAE"],
            "validation_MAE": validation_metrics["MAE"],
            "train_RMSE": train_metrics["RMSE"],
            "validation_RMSE": validation_metrics["RMSE"],
            "train_R2": train_metrics["R2"],
            "validation_R2": validation_metrics["R2"],
        })

    decision_tree_results = pd.DataFrame(fold_results)
    display(decision_tree_results.round(4))
else:
    print("Decision Tree skipped. Set RUN_DECISION_TREE = True to run it.")

,model,fold,train_rows,validation_rows,train_MAE,validation_MAE,train_RMSE,validation_RMSE,train_R2,validation_R2
0,Decision Tree,fold_1,82553,87927,7.4774,14.9853,12.9780,26.7196,0.9687,0.8850
1,Decision Tree,fold_2,170480,87088,7.6438,14.1633,13.0704,28.9375,0.9706,0.8822
2,Decision Tree,fold_3,257568,86826,7.7705,12.0268,13.3228,23.2546,0.9716,0.9061


## 5. Model 3 — Random Forest Regressor

**Purpose:** combine many trees to reduce variance compared with one tree.

**Overfitting controls:** bounded depth, minimum leaf size, fixed random state.

In [6]:
# Independent Random Forest block
RUN_RANDOM_FOREST = True
random_forest_results = None

if RUN_RANDOM_FOREST:
    fold_results = []

    for _, item in manifest.iterrows():
        train = pd.read_csv(SPLIT_DIR / item["train_file"])
        validation = pd.read_csv(SPLIT_DIR / item["validation_file"])

        columns, numeric, categorical = model_input_columns(
            train, TARGET_COLUMN, feature_columns=FEATURE_COLUMNS
        )
        X_train = train[columns]
        y_train = train[TARGET_COLUMN]
        X_validation = validation[columns]
        y_validation = validation[TARGET_COLUMN]

        pipeline = make_model_pipeline("Random Forest", numeric, categorical)
        pipeline.fit(X_train, y_train)

        train_prediction = pipeline.predict(X_train)
        validation_prediction = pipeline.predict(X_validation)
        train_metrics = regression_metrics(y_train, train_prediction)
        validation_metrics = regression_metrics(y_validation, validation_prediction)

        fold_results.append({
            "model": "Random Forest",
            "fold": item["fold"],
            "train_rows": len(train),
            "validation_rows": len(validation),
            "train_MAE": train_metrics["MAE"],
            "validation_MAE": validation_metrics["MAE"],
            "train_RMSE": train_metrics["RMSE"],
            "validation_RMSE": validation_metrics["RMSE"],
            "train_R2": train_metrics["R2"],
            "validation_R2": validation_metrics["R2"],
        })

    random_forest_results = pd.DataFrame(fold_results)
    display(random_forest_results.round(4))
else:
    print("Random Forest skipped. Set RUN_RANDOM_FOREST = True to run it.")

,model,fold,train_rows,validation_rows,train_MAE,validation_MAE,train_RMSE,validation_RMSE,train_R2,validation_R2
0,Random Forest,fold_1,82553,87927,6.2887,11.1635,11.1669,21.8413,0.9768,0.9231
1,Random Forest,fold_2,170480,87088,6.5580,11.2881,11.4207,25.2119,0.9775,0.9106
2,Random Forest,fold_3,257568,86826,6.7963,9.5057,11.7660,19.2640,0.9778,0.9356


## 6. Model 4 — HistGradientBoosting Regressor

**Purpose:** build trees sequentially so later trees focus on previous errors.

**Time-series control:** `early_stopping=False` avoids a hidden random validation split.

In [7]:
# Independent HistGradientBoosting block
RUN_HIST_GRADIENT_BOOSTING = True
hist_gradient_results = None

if RUN_HIST_GRADIENT_BOOSTING:
    fold_results = []

    for _, item in manifest.iterrows():
        train = pd.read_csv(SPLIT_DIR / item["train_file"])
        validation = pd.read_csv(SPLIT_DIR / item["validation_file"])

        columns, numeric, categorical = model_input_columns(
            train, TARGET_COLUMN, feature_columns=FEATURE_COLUMNS
        )
        X_train = train[columns]
        y_train = train[TARGET_COLUMN]
        X_validation = validation[columns]
        y_validation = validation[TARGET_COLUMN]

        pipeline = make_model_pipeline("HistGradientBoosting", numeric, categorical)
        pipeline.fit(X_train, y_train)

        train_prediction = pipeline.predict(X_train)
        validation_prediction = pipeline.predict(X_validation)
        train_metrics = regression_metrics(y_train, train_prediction)
        validation_metrics = regression_metrics(y_validation, validation_prediction)

        fold_results.append({
            "model": "HistGradientBoosting",
            "fold": item["fold"],
            "train_rows": len(train),
            "validation_rows": len(validation),
            "train_MAE": train_metrics["MAE"],
            "validation_MAE": validation_metrics["MAE"],
            "train_RMSE": train_metrics["RMSE"],
            "validation_RMSE": validation_metrics["RMSE"],
            "train_R2": train_metrics["R2"],
            "validation_R2": validation_metrics["R2"],
        })

    hist_gradient_results = pd.DataFrame(fold_results)
    display(hist_gradient_results.round(4))
else:
    print("HistGradientBoosting skipped. Set RUN_HIST_GRADIENT_BOOSTING = True to run it.")

,model,fold,train_rows,validation_rows,train_MAE,validation_MAE,train_RMSE,validation_RMSE,train_R2,validation_R2
0,HistGradientBoosting,fold_1,82553,87927,9.5299,11.2714,15.5060,22.4018,0.9553,0.9191
1,HistGradientBoosting,fold_2,170480,87088,9.5464,11.2510,15.9977,24.8792,0.9559,0.9129
2,HistGradientBoosting,fold_3,257568,86826,9.5918,9.4832,16.4544,19.4051,0.9567,0.9346


## 7. Combine completed model results

In [8]:
trained_results = [
    result for result in [
        linear_results,
        decision_tree_results,
        random_forest_results,
        hist_gradient_results,
    ]
    if result is not None and not result.empty
]

if trained_results:
    all_results = pd.concat(trained_results, ignore_index=True)

    model_summary = (
        all_results.groupby("model", as_index=False)
        .agg(
            mean_validation_MAE=("validation_MAE", "mean"),
            mean_validation_RMSE=("validation_RMSE", "mean"),
            validation_RMSE_std=("validation_RMSE", "std"),
            mean_validation_R2=("validation_R2", "mean"),
            mean_train_RMSE=("train_RMSE", "mean"),
        )
    )

    model_summary["RMSE_train_to_validation_gap"] = (
        model_summary["mean_validation_RMSE"]
        - model_summary["mean_train_RMSE"]
    )

    display(model_summary.round(4))

    AUDIT_DIR.mkdir(parents=True, exist_ok=True)
    all_results.to_csv(PHASE8_RESULTS, index=False)
    print("Created:", PHASE8_RESULTS)
else:
    all_results = None
    print("No model results available. Set the required RUN_... flags to True.")

,model,mean_validation_MAE,mean_validation_RMSE,validation_RMSE_std,mean_validation_R2,mean_train_RMSE,RMSE_train_to_validation_gap
0,Decision Tree,13.7251,26.3039,2.8641,0.8911,13.1238,13.1801
1,HistGradientBoosting,10.6685,22.2287,2.7411,0.9222,15.9860,6.2427
2,Linear Regression,13.2784,26.3649,2.4835,0.8906,23.7472,2.6177
3,Random Forest,10.6524,22.1057,2.9827,0.9231,11.4512,10.6545


Created: /Users/ruchi/Documents/FDM_Mini_Project/data/audit/phase8_all_model_results.csv


## 8. Leakage and overfitting checks

- Training and validation are separated chronologically.
- Each fold fits its own preprocessing pipeline using training rows only.
- The final 2017 holdout is not read in this notebook.
- Compare train vs validation RMSE to identify possible overfitting.
- Compare validation RMSE across folds to identify instability.